# L7 — Dati da internet e una regressione (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/L07_internet_regressione/L07_soluzioni.ipynb)

## Promemoria

- **Shift + Invio** esegue una cella. **File → Salva una copia in Drive** per non perdere il lavoro.
- Per ogni passo trovi 🔎 **cosa cercare**. Gemini (✨) è disponibile, ma prova prima a capire cosa ti serve.
- ⏱️ Bloccato da più di 10 minuti? Chiedi.
- Se fai un pasticcio: **Runtime → Riavvia ed esegui tutto**.

## Contesto

Finora abbiamo usato dati preparati per il corso. Oggi due novità:

1. **dati veri, presi da internet**: le emissioni di CO₂ e il PIL di tutti i paesi del mondo, pubblicati da [Our World in Data](https://github.com/owid/co2-data);
2. **una regressione**, lo strumento base dell'economista, usata come una "scatola nera": non serve sapere come funziona dentro, serve saperla lanciare e **leggere**.

## 0. Preparazione

Esegui questa cella così com'è.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"

## Parte 1 — Dati da internet

### 1. Scarica

Leggi il file CSV all'indirizzo `url` e salvalo in `co2`. Quante righe e quante colonne ha? Le colonne sono molte: mostra solo i loro nomi.

🔎 *Da cercare:* `pandas read_csv` · `pandas columns`

In [ ]:
co2 = pd.read_csv(url)
print(co2.shape)
print(list(co2.columns))

### 2. L'Italia

Mostra le righe dell'Italia (`country` uguale a `"Italy"`), solo le colonne anno, popolazione, PIL ed emissioni pro capite. Da che anno a che anno vanno i dati?

🔎 *Da cercare:* `pandas filtrare righe` · `pandas selezionare più colonne`

In [ ]:
italia = co2[co2["country"] == "Italy"]
print(italia["year"].min(), italia["year"].max())
italia[["year", "population", "gdp", "co2_per_capita"]].tail()

### 3. Disegnala

Disegna le emissioni di CO₂ pro capite dell'Italia dal 1950 in poi, con titolo e unità di misura (tonnellate per persona).

🔎 *Da cercare:* `pandas plot x y` · `matplotlib title ylabel`

In [ ]:
italia[italia["year"] >= 1950].plot(x="year", y="co2_per_capita", legend=False)
plt.title("Emissioni di CO2 pro capite, Italia")
plt.ylabel("Tonnellate per persona")
plt.show()

### 4. Quanta CO₂ nel mondo?

Prendi solo l'anno 2022 e somma la colonna `co2` (milioni di tonnellate). Poi guarda il valore della riga `"World"`. I due numeri tornano?

🔎 *Da cercare:* `pandas sum`

In [ ]:
anno = co2[co2["year"] == 2022]
print(anno["co2"].sum())
print(anno[anno["country"] == "World"]["co2"])

*Risposta:* No: la somma è circa sei volte il totale mondiale. La colonna `country` non contiene solo paesi, ma anche **aggregati**: continenti, Unione europea, gruppi di reddito, il mondo intero. Sommandoli, gli stessi paesi vengono contati più volte.

### 5. Solo i paesi

Gli aggregati non hanno un codice paese (`iso_code` mancante). Tieni solo le righe del 2022 con un codice paese e rifai la somma. Ora torna, più o meno, con il totale mondiale?

🔎 *Da cercare:* `pandas notna`

In [ ]:
paesi = anno[anno["iso_code"].notna()].copy()
print(len(paesi), paesi["co2"].sum())

*Risposta:* Circa 218 paesi e 36.500 milioni di tonnellate, contro le 37.500 di `World`: la piccola differenza sono le emissioni che non appartengono a nessun paese (aviazione e navigazione internazionali).

## Parte 2 — Una regressione

Una **regressione lineare** (OLS) cerca la retta che descrive meglio come una variabile cambia al variare di un'altra. Con `statsmodels` si scrive come una formula: `"y ~ x"` vuol dire *"y in funzione di x"*.

### 6. Prima guarda

Carica il dataset delle imprese e disegna un grafico a dispersione: addetti sull'asse orizzontale, fatturato su quello verticale. Che relazione vedi?

🔎 *Da cercare:* `pandas plot scatter`

In [ ]:
imprese = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/imprese_pulito.xlsx")
imprese.plot(kind="scatter", x="addetti", y="fatturato", alpha=0.3)
plt.show()

### 7. Lancia la regressione

Stima la regressione del fatturato sugli addetti e mostra il riepilogo dei risultati.

🔎 *Da cercare:* `statsmodels formula ols` · `statsmodels summary`

In [ ]:
modello = smf.ols("fatturato ~ addetti", data=imprese).fit()
print(modello.summary())

### 8. Leggila

Dal riepilogo, trova e scrivi:

1. il **coefficiente** di `addetti`: cosa vuol dire, in parole, con le unità di misura?
2. l'**R²**: quanto della variabilità del fatturato è spiegata dagli addetti?
3. il **numero di osservazioni** usate. Sono 2.500? Perché no?

🔎 *Da cercare:* `statsmodels params rsquared nobs`

In [ ]:
print(modello.params)
print(modello.rsquared)
print(modello.nobs)

*Risposta:* 1) circa 140: ogni addetto in più è associato in media a circa 140 mila euro di fatturato in più (il fatturato è in migliaia). 2) R² circa 0,73: gli addetti spiegano circa tre quarti della variabilità. 3) 2.375, non 2.500: le righe con fatturato **o** addetti mancanti vengono escluse in silenzio. Va sempre controllato e dichiarato.

### 9. Una variabile in più

Aggiungi la spesa in R&S alla regressione. Cosa succede al coefficiente degli addetti? E al numero di osservazioni?

🔎 *Da cercare:* `statsmodels formula più variabili`

In [ ]:
modello2 = smf.ols("fatturato ~ addetti + spesa_rs", data=imprese).fit()
print(modello2.params)
print(modello2.nobs)

*Risposta:* Il coefficiente degli addetti scende (da circa 140 a circa 116): parte di quello che attribuivamo agli addetti era legato alla spesa in R&S. Le osservazioni scendono a 2.255, perché ora basta che manchi uno di **tre** valori. Confrontare modelli stimati su campioni diversi richiede cautela.

## Domanda finale

La regressione dice che più addetti vanno con più fatturato. Vuol dire che **assumendo** una persona un'impresa fattura 140 mila euro in più? Perché sì o perché no?

*Risposta:* No: la regressione misura un'**associazione**, non un effetto causale. Le imprese grandi hanno più addetti e più fatturato per molti motivi insieme (mercato, settore, storia). Per parlare di cause servono disegni di ricerca appositi: è il lavoro dei vostri corsi di econometria.

---
## ⭐ Bonus (per chi ha finito)

### B1. PIL e CO₂ nel mondo

Con i dati dei paesi nel 2022, calcola il PIL pro capite (`gdp / population`) e disegna un grafico a dispersione PIL pro capite contro CO₂ pro capite, con **entrambi gli assi in scala logaritmica**.

🔎 *Da cercare:* `pandas plot scatter logx logy`

In [ ]:
paesi["pil_pc"] = paesi["gdp"] / paesi["population"]
paesi.plot(kind="scatter", x="pil_pc", y="co2_per_capita", logx=True, logy=True, alpha=0.6)
plt.xlabel("PIL pro capite (dollari internazionali, log)")
plt.ylabel("CO2 pro capite (t, log)")
plt.show()

### B2. Un'elasticità

Stima la regressione del logaritmo della CO₂ pro capite sul logaritmo del PIL pro capite. In una regressione log-log il coefficiente è un'**elasticità**: se il PIL pro capite cresce dell'1%, di quanto cresce la CO₂ pro capite?

🔎 *Da cercare:* `statsmodels formula np.log` · `elasticità regressione log log`

In [ ]:
m = smf.ols("np.log(co2_per_capita) ~ np.log(pil_pc)", data=paesi).fit()
print(m.params)
print(m.nobs)

*Risposta:* Circa 1: tra paesi, a un PIL pro capite più alto dell'1% corrispondono emissioni pro capite più alte di circa l'1%. Su 164 paesi: per gli altri manca il PIL.

---
## Compito a casa

- Nessun compito Kaggle: si avvicina il mini-progetto finale. Pensa a una domanda a cui vorresti rispondere con i dati (anche con quelli di Our World in Data).
- Facoltativo: [QuantEcon – Pandas for Panel Data](https://python-programming.quantecon.org/pandas_panel.html)